In [1]:
import glob
import numpy as np
import pandas as pd

# taxonomic name resolution
from ete3 import NCBITaxa

# Initialize NCBI taxonomy handler
ncbi = NCBITaxa()

In [2]:
## Supplement unclassified taxonomic levels with the name of the closest classified parent level
# Example: name_unclassified('d__Bacteria|k__Pseudomonadati|p__Pseudomonadota|c__unclassified')
# --> 'd__Bacteria|k__Pseudomonadati|p__Pseudomonadota|c__Pseudomonadota_unclassified'
def name_unclassified(name):
    name_list = name.split('|')[::-1]

    for ind, name_element in enumerate(name_list):
        name_element = name_element[3:]
        fl = name_element[0]

        if (fl.islower() and fl.isalpha()
            ) or 'Incertae' in name_element or 'ASV' in name_element:
            pass
        else:
            break

    if ind == 0:
        return '|'.join(name_list[::-1])

    prefix = name_element

    for ind2, name_element in enumerate(name_list):
        if ind2 == ind:
            break

        tl = name_element[0]
        name_element = name_element[3:]

        name_list[ind2] = f"{tl}__{prefix}_{name_element}"

    new_val = '|'.join(name_list[::-1])

    return new_val

SID = {}
BROKEN_NCBI_TAXA = {}

## Retrieve full taxonomic lineage from NCBI taxonomy database using NCBI taxon ID
# Example: get_full_name(562)
# Returns: 'd__Bacteria|k__Pseudomonadati|p__Pseudomonadota|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli'
def get_full_name(idn):
    try:

        ordered_orgs = ['d', 'k', 'p', 'c', 'o', 'f', 'g', 's']

        lpath = ncbi.get_lineage_translator([idn])[idn][1:]
        orgs = [
            ncbi.get_rank([ii])[ii].replace('acellular root',
                                            'domain').replace('clade',
                                                              'xxx')[0]
            for ii in lpath
        ]

        lpath_names = [ncbi.get_taxid_translator([i])[i] for i in lpath]
        d = dict(zip(orgs, lpath_names))
        d = [(o, d.get(o, 'unclassified')) for o in ordered_orgs]
        lpath_names = ['__'.join(vv) for vv in list(d)]

    except Exception as e:
        if idn not in BROKEN_NCBI_TAXA.keys():
            try:
                SID[name_unclassified(BROKEN_NCBI_TAXA.get(idn, np.nan))] = idn
            except:
                pass
        return BROKEN_NCBI_TAXA.get(idn, np.nan)

    SID[name_unclassified('|'.join(lpath_names).replace(' ', '_'))] = idn
    return '|'.join(lpath_names).replace(' ', '_')

In [3]:
# Aggregate taxonomic counts from species level to higher taxonomic levels
# Taxonomies are assumed to be delimited by '|'
# Returns a list of pandas DataFrames with count summaries for each taxonomic rank
def derive_tables(data):
    datas = []
    data = data.copy(deep=True)
    data.columns.name = 'TAXA'

    datas.append(data.copy(deep=True))

    data = data.T.reset_index()

    depth = len(data['TAXA'].values[0].split('|'))

    for _ in range(depth - 1):
        data['TAXA'] = data['TAXA'].apply(lambda x: '|'.join(x.split('|')[:-1]))
        data = data.groupby('TAXA').sum()
        data = data.T
        datas.append(data.copy(deep=True))
        data = data.T.reset_index()

    return datas

# Example:
derive_tables(pd.DataFrame([[5, 5]], columns=['d__Bacteria|k__Pseudomonadati', 'd__Bacteria|k__Bacillati']))[-1]

TAXA,d__Bacteria
0,10


In [4]:
meta = pd.read_excel('id-meta.xlsx', index_col='Sample')
meta.head()

,Group,Alias,microbiome sapmle ID
Sample,,,
VZK1-2,control,CON,int7
VZK1-4,control,CON,int8
VZK1-6,control,CON,int9
VZK2-1,control,CON,VZK2-1
VZK2-2,control,CON,VZK2-2


In [5]:
# Retrieve full file paths for Kraken2 and HUMAnN report files using glob
REPORTS = glob.glob('VZK_res/*/*k2*')
PATHS = glob.glob('VZK_res/*/*path*')
REPORTS[0]

'VZK_res/int7/int7_report_k2_standard16_v2.txt'

In [6]:
RELABS = []
COUNTS = []

# Read the Kraken2 database summary, which provides the total number of unique k-mers for each reference genome
d2 = pd.read_table('k2_standard16_standard_summary.txt')
d2.columns = [
    'pcnt', 'kmers_ref_clade', 'kmers_ref', 'order', 'ncbi_id', 'name'
]
# Build a dictionary mapping reference genome IDs to their unique k-mer counts
d2 = d2.set_index('ncbi_id')['kmers_ref'].to_dict()

for F1 in REPORTS:

    _, ID, _ = F1.split('/')
    print(f'Working with: {ID}')

    # Load Kraken2 classification report results
    d = pd.read_table(F1)
    # Kraken2 report columns (in order):
    # 1. first approx. % abundance
    # 2. reads assigned to clade
    # 3. reads assigned to taxon
    # 4. total k-mers assigned
    # 5. unique k-mers assigned
    # 6. taxonomic rank
    # 7. NCBI ID
    # 8. scientific name
    d.columns = [
        'pcnt', 'reads_clade', 'reads_taxon', 'kmers_total_aligned',
        'kmers_unique_aligned', 'order', 'ncbi_id', 'name'
    ]

    # Retain only species-level taxonomic assignments
    d = d[d.order == 'S']
    # Retrieve the full taxonomic name from NCBI using the taxon ID
    d['name'] = d['ncbi_id'].apply(get_full_name)

    # Retain only records with reads directly assigned to the taxon (excluding clade-level propagated reads)
    d = d[d.reads_taxon > 0]
    # Store the total number of classified reads (taxonomic table depth) before count adjustment
    DEPTH = d['reads_taxon'].sum()

    # Add column specifying the number of unique k-mers in the reference genome to which reads were aligned
    d['kmers_ref'] = d.ncbi_id.map(d2)
    d = d[d.kmers_ref > 100]

    ## Calculate E-value filter:
    
    # E-value = (K / R) × C
    # where:
    #   K = unique k-mers assigned to taxon
    #   R = reads assigned to taxon
    #   C = reference genome coverage = unique k-mers aligned / total unique k-mers in reference genome
    
    # Reference genome coverage (C) = unique k-mers aligned to the taxon / total unique k-mers in the reference genome
    d['coverage'] = d['kmers_unique_aligned'] / (d['kmers_ref'])
    # Multiply the proportion of informative reads (K/R) by reference genome coverage (C) to obtain the E-value
    d['Evalue'] = (d['kmers_unique_aligned'] /
                   d['reads_taxon']) * d['coverage']

    # Keep only records with E-value > 0.1
    d = d[d.Evalue > 0.15]

    # Adjust number of aligned read counts by reference genome size (approx. by unique k-mer count)
    d['reads_taxon_adj'] = d['reads_taxon'] / (d['kmers_ref'])
    d['reads_taxon_adj'] /= d['reads_taxon_adj'].sum()
    d['reads_taxon_adj'] = d['reads_taxon_adj'] * DEPTH

    # Format the taxonomic record for downstream analysis
    d = d[d.name.notna()]
    d = pd.DataFrame(d.set_index('name')['reads_taxon_adj']).T.filter(like='Bacteria')

    # Calculate total sums
    datas = derive_tables(d)
    COUNT = pd.concat(datas[::-1],
                      axis=1).T.reset_index().groupby('TAXA').sum().T
    
    # Normalize counts using TSS (total sum scaling):
    RELAB = pd.concat(
        [d.div(d.sum(axis=1), axis=0) * 100 for d in datas[::-1]], axis=1)
    
    # If any exist, name unclassified levels using the name_unclassified() function
    COUNT.columns = [name_unclassified(c) for c in COUNT.columns]
    RELAB.columns = [name_unclassified(c) for c in RELAB.columns]
    
    # Save sample data
    RELAB.index = [ID]
    RELABS.append(RELAB)

    COUNT.index = [ID]
    COUNTS.append(COUNT)

Working with: int7
Working with: int8
Working with: int9
Working with: VZK2-1
Working with: VZK2-2
Working with: VZK2-3
Working with: VZK2-4
Working with: VZK2-5
Working with: VZK2-6
Working with: VZK3-2
Working with: VZK3-3
Working with: VZK3-5
Working with: VZK3-6
Working with: VZK4-10
Working with: VZK4-11
Working with: VZK4-12
Working with: VZK4-8
Working with: VZK4-9
Working with: VZK5-1
Working with: VZK5-3
Working with: VZK5-4
Working with: VZK5-5
Working with: VZK5-6
Working with: VZK6-13
Working with: VZK6-14
Working with: VZK6-16
Working with: VZK6-18


In [7]:
# Combine ndividual sample data into a single DataFrame
RELAB = pd.concat(RELABS).fillna(0)
COUNTS = pd.concat(COUNTS).fillna(0)

RELAB = RELAB.loc[meta['microbiome sapmle ID']]
RELAB.index = meta.index

COUNTS = COUNTS.loc[meta['microbiome sapmle ID']]
COUNTS.index = meta.index

In [8]:
# Display median relative abundance at the phylum level
RELAB.filter(regex=f'p__[A-z0-9-.]+$').median().sort_values(ascending=False).head(10)

d__Bacteria|k__Bacillati|p__Bacillota                       61.977507
d__Bacteria|k__Pseudomonadati|p__Bacteroidota               26.731211
d__Bacteria|k__Pseudomonadati|p__Campylobacterota            3.487226
d__Bacteria|k__Bacillati|p__Actinomycetota                   1.824034
d__Bacteria|k__Pseudomonadati|p__Pseudomonadota              0.000138
d__Bacteria|k__Pseudomonadati|p__Chlamydiota                 0.000000
d__Bacteria|k__Pseudomonadati|p__Spirochaetota               0.000000
d__Bacteria|k__Bacillati|p__Chloroflexota                    0.000000
d__Bacteria|k__Bacillati|p__Mycoplasmatota                   0.000000
d__Bacteria|k__Pseudomonadati|p__Thermodesulfobacteriota     0.000000
dtype: float64

In [9]:
# Display median relative abundance at the family level
RELAB.filter(regex=f'f__[A-z0-9-.]+$').median().sort_values(ascending=False).head(10)

d__Bacteria|k__Bacillati|p__Bacillota|c__Bacilli|o__Lactobacillales|f__Lactobacillaceae                                 29.492691
d__Bacteria|k__Bacillati|p__Bacillota|c__Clostridia|o__Peptostreptococcales|f__Peptostreptococcaceae                     5.982599
d__Bacteria|k__Pseudomonadati|p__Bacteroidota|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae                          4.547811
d__Bacteria|k__Pseudomonadati|p__Campylobacterota|c__Epsilonproteobacteria|o__Campylobacterales|f__Helicobacteraceae     3.081566
d__Bacteria|k__Pseudomonadati|p__Bacteroidota|c__Bacteroidia|o__Bacteroidales|f__Muribaculaceae                          2.926768
d__Bacteria|k__Bacillati|p__Actinomycetota|c__Coriobacteriia|o__Eggerthellales|f__Eggerthellaceae                        1.582013
d__Bacteria|k__Bacillati|p__Bacillota|c__Bacilli|o__Lactobacillales|f__Streptococcaceae                                  1.102840
d__Bacteria|k__Pseudomonadati|p__Bacteroidota|c__Bacteroidia|o__Bacteroidales|f__Tannerell

In [10]:
# Save taxonomic profiling data

%mkdir -p prodata/
RELAB.to_csv('prodata/data-relab.tsv.gz', sep='\t', compression='gzip')
COUNTS.to_csv('prodata/data-counts.tsv.gz', sep='\t', compression='gzip')

In [11]:
# Load HUMAnN profiling reports and retain only mapped functional records
# Convert CPM values to relative abundance (%) using TSS normalization
RELABS = []

for F2 in PATHS:

    _, ID, _ = F2.split('/')

    row = pd.read_table(F2).iloc[2:].set_index(
        '# Pathway HUMAnN v4.0.0.alpha.2').T
    row.index = [ID]
    RELABS.append(row)

RELAB = pd.concat(RELABS)
RELAB = (RELAB.div(RELAB.sum(axis=1), axis=0) * 100).fillna(0)

In [12]:
# Display median relative abundance of top 20 functions
RELAB.median().sort_values(ascending=False).head(20)

# Pathway HUMAnN v4.0.0.alpha.2
TRNA-CHARGING-PWY: tRNA charging                                          1.655635
PWY-1042: glycolysis IV                                                   1.400938
GLUCONEO-PWY: gluconeogenesis I                                           1.375052
ANAGLYCOLYSIS-PWY: glycolysis III (from glucose)                          1.340744
PWY-5941: glycogen degradation II                                         1.148965
CALVIN-PWY: Calvin-Benson-Bassham cycle                                   1.108708
PWY-6527: stachyose degradation                                           1.088832
GLYCOLYSIS: glycolysis I (from glucose 6-phosphate)                       1.055914
PWY-6700: queuosine biosynthesis I (de novo)                              0.983228
GLUCOSE1PMETAB-PWY: glucose and glucose-1-phosphate degradation           0.976276
PWY-7185: UTP and CTP dephosphorylation I                                 0.969497
PWY-7221: guanosine ribonucleotides de novo biosynthesi

In [13]:
# Save functional inference data
RELAB = RELAB.loc[meta['microbiome sapmle ID']]
RELAB.index = meta.index
RELAB.to_csv('prodata/path-relab.tsv.gz', sep='\t', compression='gzip')